In [13]:
import numpy as np
import pandas as pd
import scipy.stats as stats

In [14]:
!bcftools query -f '[%GT\t]\n' ~/public/1000Genomes/1000G_chr22_pruned.vcf.gz > ~/private/chr22_genotypes.txt

In [15]:
def load_vcf_genotypes(filename):
    df = pd.read_csv(filename, sep='\t', header=None, low_memory=False)
    
    #function to convert '0|1' or '1/1' to an integer (0, 1, or 2)
    def parse_gt(gt_string):
        if pd.isna(gt_string): return 0
        # Replace | with / to handle both phased and unphased
        alleles = str(gt_string).replace('|', '/').split('/')
        try:
            return int(alleles[0]) + int(alleles[1])
        except (ValueError, IndexError):
            return 0

    #apply the converter to all columns except the last one if it's empty
    genotypes = df.iloc[:, :-1].applymap(parse_gt).values
    return genotypes

In [16]:
G = load_vcf_genotypes('chr22_genotypes.txt')
print(f"Loaded genotype matrix with shape: {G.shape}")

num_snps, num_samples = G.shape

Loaded genotype matrix with shape: (13629, 2504)


benchmarking scenario 1: polygenic architecture

In [17]:
#select 5% of SNPs to be causal
perc_causal = 0.05
causal_idx = np.random.choice(num_snps, int(num_snps * perc_causal), replace=False)

#assign effect sizes (beta) and generate phenotype Y
true_betas = np.zeros(num_snps)
true_betas[causal_idx] = np.random.normal(0, 1, len(causal_idx))
genetic_signal = np.dot(G.T, true_betas)
noise = np.random.normal(0, np.std(genetic_signal), num_samples) # h2 ~ 0.5
Y_polygenic = genetic_signal + noise

benchmarking scenario 2: pure inflation (confounding)

In [18]:
#no genetic signal->all betas = 0, only random noise plus a bias
Y_inflated = np.random.normal(0, 1, num_samples)
inflation_shift = 0.5 #simulates population stratification/bias

In [19]:
def run_mini_gwas(phenotype, genotypes):
    results = []
    for i in range(num_snps):
        snp = genotypes[i, :]
        slope, intercept, r_val, p_val, std_err = stats.linregress(snp, phenotype)
        results.append(['rs_sim_' + str(i), slope, std_err, p_val])
    return pd.DataFrame(results, columns=['MarkerName', 'Beta', 'SE', 'Pvalue'])

In [20]:
df_poly = run_mini_gwas(Y_polygenic, G)
df_poly.to_csv('sim_polygenic.tsv', sep='\t', index=False)

df_infl = run_mini_gwas(Y_inflated, G)
#manually inflate chi-sq statistics for inflated scenario
df_infl['Pvalue'] = df_infl['Pvalue'] * 0.1 #simplistic way to simulate bias
df_infl.to_csv('sim_inflated.tsv', sep='\t', index=False)